## Dataset cleanup

The original dataset is loaded from the Excel file. Since the column names start in the fourth row of the file, the `header=3` parameter is used during import. The column names are standardized, empty non-patient rows are removed and the cleaned dataset is saved as a CSV file for the following preprocessing steps.

In [40]:
import pandas as pd

# load dataset
df = pd.read_excel("data/raw/dataset.xlsx", header=3)

# clean up column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# remove rows that do not have a patient ID
df = df.dropna(subset=["patient"])

# sanity checks
display(df.head())
df.info()

# save cleaned dataset as a new CSV file
df.to_csv("data/processed/dataset_clean.csv", index=False)

,patient,gender,age_(y),schooling_(y),breastfeeding,varicella,initial_symptom,mono_or_polysymptomatic,oligoclonal_bands,llssep,ulssep,vep,baep,periventricular_mri,cortical_mri,infratentorial_mri,spinal_cord_mri,initial_edss,final_edss,group
0,1.0,1,34.0,20.0,1,1,2,1,0,1,1,0,0,0,1,0,1,1.0,1.0,1
1,2.0,1,61.0,25.0,3,2,10,2,1,1,0,1,0,0,0,0,1,2.0,2.0,1
2,3.0,1,22.0,20.0,3,1,3,1,1,0,0,0,0,0,1,0,0,1.0,1.0,1
3,4.0,2,41.0,15.0,1,1,7,2,1,0,1,1,0,1,1,0,0,1.0,1.0,1
4,5.0,2,34.0,20.0,2,1,6,2,0,1,0,0,0,1,0,0,0,1.0,1.0,1


<class 'pandas.DataFrame'>
RangeIndex: 273 entries, 0 to 272
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   patient                  273 non-null    float64
 1   gender                   273 non-null    object 
 2   age_(y)                  273 non-null    float64
 3   schooling_(y)            272 non-null    float64
 4   breastfeeding            273 non-null    object 
 5   varicella                273 non-null    object 
 6   initial_symptom          272 non-null    object 
 7   mono_or_polysymptomatic  273 non-null    object 
 8   oligoclonal_bands        273 non-null    object 
 9   llssep                   273 non-null    object 
 10  ulssep                   273 non-null    object 
 11  vep                      273 non-null    object 
 12  baep                     273 non-null    object 
 13  periventricular_mri      273 non-null    object 
 14  cortical_mri             273 non-null

### Data inspection

The cleaned CSV file is loaded and inspected before further preprocessing. The dataset shape, target variable distribution, and data types are checked to ensure that the data was saved and loaded correctly.

In [41]:
# load cleaned csv file
df = pd.read_csv("data/processed/dataset_clean.csv")

# check shape of the dataset
print("Dataset shape:", df.shape)

# check distribution of the target variable (group)
print("\nTarget variable distribution:")
print(df["group"].value_counts(dropna=False))

# check data types of the columns
print("\nData types:")
print(df.dtypes)

Dataset shape: (273, 20)

Target variable distribution:
group
2    148
1    125
Name: count, dtype: int64

Data types:
patient                    float64
gender                       int64
age_(y)                    float64
schooling_(y)              float64
breastfeeding                int64
varicella                    int64
initial_symptom            float64
mono_or_polysymptomatic      int64
oligoclonal_bands            int64
llssep                       int64
ulssep                       int64
vep                          int64
baep                         int64
periventricular_mri          int64
cortical_mri                 int64
infratentorial_mri           int64
spinal_cord_mri              int64
initial_edss               float64
final_edss                 float64
group                        int64
dtype: object


## Data preprocessing

In this section, the cleaned CSV file is loaded and prepared for machine learning. This includes checking the target variable, converting coded variables to numeric values, handling missing values, excluding leakage variables, and preparing the data for model training.

### Target variable

The target variable is derived from the `group` column. According to the dataset encoding, `1` represents patients who converted to clinically definite multiple sclerosis (CDMS), while `2` represents patients who remained non-CDMS. For binary classification, the target is recoded so that conversion to CDMS is represented as `1` and no conversion as `0`.

In [42]:
# create copy of the dataset for preprocessing to keep original cleaned dataframe intact
df_model = df.copy()

# recode target variable for binary classification
df_model["conversion"] = df_model["group"].map({
    1: 1,
    2: 0
})

# check if target variable was created correctly
print("Target variable distribution:")
print(df_model["conversion"].value_counts(dropna=False))

Target variable distribution:
conversion
0    148
1    125
Name: count, dtype: int64


### Feature and target separation

The dataset is separated into input features (`X`) and the target variable (`y`). Columns that should not be used for prediction are excluded. The patient identifier is removed because it does not contain clinical information. The original `group` column is removed because it has already been transformed into the binary target variable. The EDSS columns are excluded to avoid data leakage, since EDSS values are only available for patients who converted to CDMS.

In [43]:
# define target variable y
y = df_model["conversion"]

# drop columns that are not predictors and create feature matrix X
columns_to_drop = [
    "patient",
    "group",
    "conversion",
    "initial_edss",
    "final_edss"
]

X = df_model.drop(columns=columns_to_drop)

# check shape of feature matrix and target vector
print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

# sanity check of feature columns
print("\nUpdated feature columns:")
print(X.columns.tolist())
display(X.head())

Feature matrix shape: (273, 16)
Target vector shape: (273,)

Updated feature columns:
['gender', 'age_(y)', 'schooling_(y)', 'breastfeeding', 'varicella', 'initial_symptom', 'mono_or_polysymptomatic', 'oligoclonal_bands', 'llssep', 'ulssep', 'vep', 'baep', 'periventricular_mri', 'cortical_mri', 'infratentorial_mri', 'spinal_cord_mri']


,gender,age_(y),schooling_(y),breastfeeding,varicella,initial_symptom,mono_or_polysymptomatic,oligoclonal_bands,llssep,ulssep,vep,baep,periventricular_mri,cortical_mri,infratentorial_mri,spinal_cord_mri
0,1,34.0,20.0,1,1,2.0,1,0,1,1,0,0,0,1,0,1
1,1,61.0,25.0,3,2,10.0,2,1,1,0,1,0,0,0,0,1
2,1,22.0,20.0,3,1,3.0,1,1,0,0,0,0,0,1,0,0
3,2,41.0,15.0,1,1,7.0,2,1,0,1,1,0,1,1,0,0
4,2,34.0,20.0,2,1,6.0,2,0,1,0,0,0,1,0,0,0


### Handling unknown values

Some variables use specific numeric codes to represent unknown values. These values are replaced with missing values so that they are not treated as meaningful numerical categories by the model.

In [ ]:
import numpy as np

# create a copy of the feature matrix for preprocessing to keep original data intact
X = X.copy()

# define encoding of unknown values in dataset
unknown_value_codes = {
    "breastfeeding": 3,              # 3 = unknown
    "varicella": 3,                  # 3 = unknown
    "mono_or_polysymptomatic": 3,    # 3 = unknown
    "oligoclonal_bands": 2           # 2 = unknown
}

# Replace unknown value codes with NaN to indicate missing values in the dataset
for column, unknown_code in unknown_value_codes.items():
    X[column] = X[column].replace(unknown_code, np.nan)

Unknown value counts before replacement:
breastfeeding: 85
varicella: 45
mono_or_polysymptomatic: 6
oligoclonal_bands: 11

Missing values after replacement:
schooling_(y)               1
breastfeeding              85
varicella                  45
initial_symptom             1
mono_or_polysymptomatic     6
oligoclonal_bands          11
dtype: int64


### Recoding binary variables

Some binary variables are recoded into clearer `0` and `1` values while keeping their original column names. This makes them easier to use in the logistic regression model. The recoding is documented because the meaning of some numeric values changes compared to the original dataset encoding.

In [45]:
# gender:
# original encoding 1 = male, 2 = female
# new encoding 0 = male, 1 = female
X["gender"] = X["gender"].map({
    1: 0,
    2: 1
})

# breastfeeding:
# original encoding 1 = yes, 2 = no, 3 = unknown (already replaced with NaN)
# new encoding 1 = yes, 0 = no
X["breastfeeding"] = X["breastfeeding"].map({
    1: 1,
    2: 0
})

# varicella:
# original encoding 1 = positive, 2 = negative, 3 = unknown (already replaced with NaN)
# new encoding: 1 = positive, 0 = negative
X["varicella"] = X["varicella"].map({
    1: 1,
    2: 0
})

# mono_or_polysymptomatic:
# original encoding 1 = monosymptomatic, 2 = polysymptomatic, 3 = unknown (already replaced with NaN)
# new encoding 1 = polysymptomatic, 0 = monosymptomatic
X["mono_or_polysymptomatic"] = X["mono_or_polysymptomatic"].map({
    1: 0,
    2: 1
})

# sanity check after recoding
print(X.shape)
print("\nUpdated feature columns:")
print(X.columns.tolist())
display(X.head())

(273, 16)

Updated feature columns:
['gender', 'age_(y)', 'schooling_(y)', 'breastfeeding', 'varicella', 'initial_symptom', 'mono_or_polysymptomatic', 'oligoclonal_bands', 'llssep', 'ulssep', 'vep', 'baep', 'periventricular_mri', 'cortical_mri', 'infratentorial_mri', 'spinal_cord_mri']


,gender,age_(y),schooling_(y),breastfeeding,varicella,initial_symptom,mono_or_polysymptomatic,oligoclonal_bands,llssep,ulssep,vep,baep,periventricular_mri,cortical_mri,infratentorial_mri,spinal_cord_mri
0,0,34.0,20.0,1.0,1.0,2.0,0.0,0.0,1,1,0,0,0,1,0,1
1,0,61.0,25.0,NaN,0.0,10.0,1.0,1.0,1,0,1,0,0,0,0,1
2,0,22.0,20.0,NaN,1.0,3.0,0.0,1.0,0,0,0,0,0,1,0,0
3,1,41.0,15.0,1.0,1.0,7.0,1.0,1.0,0,1,1,0,1,1,0,0
4,1,34.0,20.0,0.0,1.0,6.0,1.0,0.0,1,0,0,0,1,0,0,0


### Transforming intital symptom

The `initial_symptom` variable contains coded symptom categories and combinations. Since these codes do not represent an ordered numerical scale, the variable is transformed into separate binary symptom indicators for visual, sensory, motor, and other symptoms. This makes the variables easier to interpret clinically and avoids treating the original codes as continuous numerical values.

In [46]:
symptom_mapping = {
    1:  {"visual": 1, "sensory": 0, "motor": 0, "other": 0},
    2:  {"visual": 0, "sensory": 1, "motor": 0, "other": 0},
    3:  {"visual": 0, "sensory": 0, "motor": 1, "other": 0},
    4:  {"visual": 0, "sensory": 0, "motor": 0, "other": 1},
    5:  {"visual": 1, "sensory": 1, "motor": 0, "other": 0},
    6:  {"visual": 1, "sensory": 0, "motor": 1, "other": 0},
    7:  {"visual": 1, "sensory": 0, "motor": 0, "other": 1},
    8:  {"visual": 0, "sensory": 1, "motor": 1, "other": 0},
    9:  {"visual": 0, "sensory": 1, "motor": 0, "other": 1},
    10: {"visual": 0, "sensory": 0, "motor": 1, "other": 1},
    11: {"visual": 1, "sensory": 1, "motor": 1, "other": 0},
    12: {"visual": 1, "sensory": 1, "motor": 0, "other": 1},
    13: {"visual": 1, "sensory": 0, "motor": 1, "other": 1},
    14: {"visual": 0, "sensory": 1, "motor": 1, "other": 1},
    15: {"visual": 1, "sensory": 1, "motor": 1, "other": 1},
}

# create new binary symptom columns
X["symptom_visual"] = X["initial_symptom"].map(
    lambda value: symptom_mapping.get(value, {}).get("visual")
)

X["symptom_sensory"] = X["initial_symptom"].map(
    lambda value: symptom_mapping.get(value, {}).get("sensory")
)

X["symptom_motor"] = X["initial_symptom"].map(
    lambda value: symptom_mapping.get(value, {}).get("motor")
)

X["symptom_other"] = X["initial_symptom"].map(
    lambda value: symptom_mapping.get(value, {}).get("other")
)

# drop  original initial_symptom column
X = X.drop(columns=["initial_symptom"])

# sanity check
print(X.shape)

print("\nNew symptom columns:")
print(X[["symptom_visual", "symptom_sensory", "symptom_motor", "symptom_other"]].head())

print("\nUpdated feature columns:")
print(X.columns.tolist())

(273, 19)

New symptom columns:
   symptom_visual  symptom_sensory  symptom_motor  symptom_other
0             0.0              1.0            0.0            0.0
1             0.0              0.0            1.0            1.0
2             0.0              0.0            1.0            0.0
3             1.0              0.0            0.0            1.0
4             1.0              0.0            1.0            0.0

Updated feature columns:
['gender', 'age_(y)', 'schooling_(y)', 'breastfeeding', 'varicella', 'mono_or_polysymptomatic', 'oligoclonal_bands', 'llssep', 'ulssep', 'vep', 'baep', 'periventricular_mri', 'cortical_mri', 'infratentorial_mri', 'spinal_cord_mri', 'symptom_visual', 'symptom_sensory', 'symptom_motor', 'symptom_other']


### Missing Value Overview

After recoding the variables, the remaining missing values are inspected. This step helps identify which features require imputation before model training.

In [47]:
missing_values = X.isna().sum()

# display columns with at least one missing value
print("Missing values per column:")
print(missing_values[missing_values > 0])

Missing values per column:
schooling_(y)               1
breastfeeding              85
varicella                  45
mono_or_polysymptomatic     6
oligoclonal_bands          11
symptom_visual              1
symptom_sensory             1
symptom_motor               1
symptom_other               1
dtype: int64


### Train-test split

The dataset is split into training and test data before imputation and scaling. This prevents information from the test set from influencing the preprocessing steps. A stratified split is used so that both sets keep a similar proportion of converted and non-converted patients.

In [48]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2, # 20% of data for testing
    stratify=y, # keep class distribution similar in train and test sets
    random_state=42 # reproducibility
)

# check resulting shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

# check whether class distribution is similar in train and test sets
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

X_train shape: (218, 19)
X_test shape: (55, 19)
y_train shape: (218,)
y_test shape: (55,)

Training target distribution:
conversion
0    0.541284
1    0.458716
Name: proportion, dtype: float64

Test target distribution:
conversion
0    0.545455
1    0.454545
Name: proportion, dtype: float64


### Handling missing values

Missing values are filled after the train-test split to avoid data leakage. For continuous variables, missing values are replaced with the median value calculated from the training set. For binary variables, missing values are replaced with the most frequent value in the training set. The same replacement values are then applied to the test set.

In [49]:
# create copies of  training and test sets
X_train_processed = X_train.copy()
X_test_processed = X_test.copy()

# define continuous variables.
continuous_features = [
    "age_(y)",
    "schooling_(y)"
]

# all other columns are binary
binary_features = [
    col for col in X_train_processed.columns
    if col not in continuous_features
]

# fill missing values in continuous variables with median from the training set
continuous_medians = X_train_processed[continuous_features].median()

X_train_processed[continuous_features] = X_train_processed[continuous_features].fillna(continuous_medians)
X_test_processed[continuous_features] = X_test_processed[continuous_features].fillna(continuous_medians)

# fill missing values in binary variables with the most frequent value from the training set
binary_modes = X_train_processed[binary_features].mode().iloc[0]

X_train_processed[binary_features] = X_train_processed[binary_features].fillna(binary_modes)
X_test_processed[binary_features] = X_test_processed[binary_features].fillna(binary_modes)

# check whether all missing values were handled
print("Missing values in training data:", X_train_processed.isna().sum().sum())
print("Missing values in test data:", X_test_processed.isna().sum().sum())

Missing values in training data: 0
Missing values in test data: 0


### Scaling continuous variables

The continuous variables `age_(y)` and `schooling_(y)` are standardized after imputation. The scaler is fitted only on the training data and then applied to both training and test data to avoid data leakage. Binary variables are not scaled.

In [50]:
from sklearn.preprocessing import StandardScaler

# define continuous variables and create a scaler object
continuous_features = [
    "age_(y)",
    "schooling_(y)"
]

scaler = StandardScaler()

# fit the scaler only on the training data
scaler.fit(X_train_processed[continuous_features])

# apply the same scaling to the training and test data
X_train_processed[continuous_features] = scaler.transform(
    X_train_processed[continuous_features]
)

X_test_processed[continuous_features] = scaler.transform(
    X_test_processed[continuous_features]
)

# check the processed training data.
display(X_train_processed.head())

# check shapes
print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)

,gender,age_(y),schooling_(y),breastfeeding,varicella,mono_or_polysymptomatic,oligoclonal_bands,llssep,ulssep,vep,baep,periventricular_mri,cortical_mri,infratentorial_mri,spinal_cord_mri,symptom_visual,symptom_sensory,symptom_motor,symptom_other
258,1,0.130954,-0.808426,1.0,1.0,0.0,0.0,0,0,0,0,0,0,0,0,1.0,0.0,0.0,0.0
73,0,-1.641637,-0.076140,1.0,1.0,1.0,1.0,0,0,0,0,1,0,1,0,0.0,0.0,1.0,1.0
67,0,-0.148928,1.144337,1.0,1.0,1.0,0.0,1,1,1,0,1,0,1,1,1.0,1.0,1.0,1.0
209,1,-0.055634,-0.808426,1.0,0.0,1.0,0.0,0,0,0,0,0,1,0,0,1.0,1.0,0.0,0.0
128,1,2.370017,-0.808426,1.0,1.0,0.0,0.0,1,1,0,1,0,1,1,1,0.0,1.0,1.0,0.0


X_train_processed shape: (218, 19)
X_test_processed shape: (55, 19)
